In [0]:

from pyspark.sql import functions as F

BUCKET_NAME='retail-de-project-27'
CATALOG = "retail_de_27"
FILE_PATTH= 's3://'+BUCKET_NAME+'/raw/orders/*.csv'
REQUIRED_COLUMNS={ "order_id",
    "customer_id",
    "product_id",
    "quantity",
    "price",
    "order_date"}

df = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(FILE_PATTH)
)

missing=REQUIRED_COLUMNS-set(df.columns)
if missing:
    raise ValueError(f"Missing required columns: {missing}")
print("Schema validation passed")

bronze_df=df.withColumn(
    'ingestion_time', F.current_timestamp()
).withColumn(
    'source_file', F.col("_metadata.file_path")
)

spark.sql(
    f"CREATE SCHEMA IF NOT EXISTS "
    f"`{CATALOG}`.`bronze`"
)
table_name = f"{CATALOG}.bronze.orders"
bronze_df.write.format('delta').mode("overwrite").saveAsTable(table_name)
display(bronze_df)

In [0]:

#verify order table is created or not

table_namre=f"{CATALOG}.bronze.orders"

saved_df=spark.table(table_name)
display(saved_df)
print("Total Counts:",saved_df.count())
saved_df.printSchema()